# Notebook 02 — Trend Analysis: Fronts, KE and EKE by Sector

**Kinetic Energy Trends and Eddy Saturation in the Southern Ocean**
Cristina Martí-Solana, Simón Ruiz, Bàrbara Barceló-Llull, Vincent Combes and Ananda Pascual

---

Long-term trends in frontal position, envelope width, and area-weighted KE/EKE, computed by ocean-basin sector from the monthly half-power point outputs of Notebook 01.

Workflow:

1. **Load** all monthly half-power point NetCDF files into one DataFrame (front detections) plus gridded KE/EKE fields
2. **Aggregate** into monthly sector-level time series (median frontal latitude, envelope width, peak value, front count; area-weighted mean KE/EKE)
3. **Fit** trends: Theil–Sen slopes with Modified Mann–Kendall (Yue–Wang) significance
4. **Test** the eddy saturation hypothesis per sector
5. **Save** all tables to `outputs/trends/` for the figure notebooks

> `energy_field_timeseries.csv` is produced by `scripts/ke_trends_analysis.py`; this notebook does not overwrite it.


## Methods: Trend Estimation

**Theil–Sen estimator** — robust median of all pairwise slopes; 95% confidence intervals from `scipy.stats.theilslopes`.

**Modified Mann–Kendall test** — non-parametric significance with the Yue and Wang (2004) autocorrelation correction (`pymannkendall`); returns Kendall's τ and a p-value corrected for serial dependence.

Significance is assessed at p < 0.05 (two-sided). Trend magnitude is always the Theil–Sen slope; significance is always the modified MK p-value.

For frontal-latitude trends, the monthly climatology is removed (deseasonalisation) before fitting, to reduce high-frequency noise.


## 1. Setup and Imports

In [1]:
import os
import sys
import warnings
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Repository paths
REPO_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from utils.regions_and_masks import compute_acc_mask_from_ssh, filter_fronts_by_acc_mask

INPUT_DIR = REPO_ROOT / "outputs" / "monthly_half-power_points"
OUTPUT_DIR = REPO_ROOT / "outputs" / "trends"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Input directory: outputs/monthly_half-power_points
Output directory: outputs/trends


## 2. Ocean-Basin Sectors

Three Southern Ocean sectors by longitude — Atlantic [290, 20)°E, Indian [20, 147)°E, Pacific [147, 290)°E — following standard Southern Ocean basin definitions.

In [2]:
# Regional sectors — three Southern Ocean basins (Orsi et al. 1995; Constable et al. 2014)
ZHANG_SECTORS = {
    "Indian":   (20,  147),   # 20°E – 147°E
    "Pacific":  (147, 290),   # 147°E – 70°W  (0–360 coords)
    "Atlantic": (290, 380),   # 70°W – 20°E   (wraps through 360°)
}

LAT_MIN, LAT_MAX = -65.0, -35.0

# ── ACC SSH-contour mask (Sokolov & Rintoul 2009) ──────────────────────────
# When USE_ACC_MASK = True the analysis is restricted to grid cells and front
# detections that fall within the ACC band, defined by mean-ADT contours.
# Adjust SSH_ACC_SOUTH / SSH_ACC_NORTH to move the southern/northern boundary.
#
#   Typical CMEMS DUACS values (all-sat, 2000–2015):
#     SSH_ACC_SOUTH ≈ −0.6 m  → near SACCF / Southern Boundary
#     SSH_ACC_NORTH ≈  0.2 m  → north of the Sub-Antarctic Front (SAF)
#
#   Reference: Sokolov & Rintoul (2009), JGR-Oceans, doi:10.1029/2008JC005248
USE_ACC_MASK  = True
SSH_ACC_SOUTH = -0.6   # m  (southern ACC boundary contour)
SSH_ACC_NORTH =  0.2   # m  (northern ACC boundary contour)

print("Sector definitions:")
for name, (lo, hi) in ZHANG_SECTORS.items():
    print(f"  {name}: [{lo}°, {hi}°)")
print(f"\nACC SSH mask: {'ENABLED' if USE_ACC_MASK else 'DISABLED'} "
      f"({SSH_ACC_SOUTH} m ≤ ADT ≤ {SSH_ACC_NORTH} m)")


Sector definitions:
  Indian: [20°, 147°)
  Pacific: [147°, 290°)
  Atlantic: [290°, 380°)

ACC SSH mask: ENABLED (-0.6 m ≤ ADT ≤ 0.2 m)


## 3. Helper Functions

Theil–Sen and Modified Mann–Kendall wrappers, longitude/sector utilities.

In [3]:
def lon_in_sector(lon, lo, hi):
    """Check if longitude (degE, +/-180) falls inside [lo, hi), handling wrap."""
    lon360 = lon % 360
    lo360 = lo % 360
    hi360 = hi % 360
    if lo360 < hi360:
        return lo360 <= lon360 < hi360
    else:
        return lon360 >= lo360 or lon360 < hi360


def assign_sector(lon, sectors):
    """Return the sector name for a given longitude."""
    for name, (lo, hi) in sectors.items():
        if lon_in_sector(lon, lo, hi):
            return name
    return "Other"


def effective_sample_size(y):
    """Lag-1 autocorrelation effective sample size, bounded to [3, n]."""
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)
    if n < 3:
        return float(n)
    r1 = np.corrcoef(y[:-1], y[1:])[0, 1]
    if not np.isfinite(r1):
        r1 = 0.0
    r1 = float(np.clip(r1, -0.99, 0.99))
    n_eff = n * (1.0 - r1) / (1.0 + r1)
    return float(np.clip(n_eff, 3.0, float(n)))


def _basic_mann_kendall(y):
    """Fallback MK test if pymannkendall is unavailable."""
    n = len(y)
    s = 0
    for k in range(n - 1):
        for j in range(k + 1, n):
            s += np.sign(y[j] - y[k])
    var_s = n * (n - 1) * (2 * n + 5) / 18.0
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0
    p = 2 * stats.norm.sf(np.abs(z))
    tau = s / (n * (n - 1) / 2.0)
    return float(tau), float(p)


def mann_kendall(y):
    """Modified MK (Yue-Wang) via pymannkendall when available; fallback to basic MK.
    Returns: tau, p_value
    """
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    if len(y) < 4:
        return np.nan, np.nan
    try:
        import pymannkendall as mk  # local import keeps notebook runnable without package
        res = mk.yue_wang_modification_test(y)
        return float(res.Tau), float(res.p)
    except Exception:
        return _basic_mann_kendall(y)


def theil_sen_slope(x, y):
    """Theil-Sen slope with CI and SE (effective sample-size corrected).
    Returns: slope, intercept, slope_se, lo_slope, hi_slope
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(y) & np.isfinite(x)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    xm, ym = x[mask], y[mask]
    res = stats.theilslopes(ym, xm, alpha=0.95)

    yhat = res.slope * xm + res.intercept
    resid = ym - yhat
    n_eff = effective_sample_size(resid)
    dof = max(n_eff - 2.0, 1.0)
    sxx = np.sum((xm - np.mean(xm)) ** 2)
    if sxx <= 0:
        slope_se = np.nan
    else:
        sigma_resid = np.sqrt(np.sum(resid ** 2) / dof)
        slope_se = sigma_resid / np.sqrt(sxx)

    return float(res.slope), float(res.intercept), float(slope_se), float(res.low_slope), float(res.high_slope)


print("Helper functions defined.")

Helper functions defined.


## 4. Load Monthly Half-Power Point Outputs

Aggregate every monthly NetCDF from Notebook 01 into a front-detection DataFrame plus gridded KE/EKE monthly fields.

In [4]:
def load_all_monthly_outputs(input_dir):
    """Load every monthly half-power point NetCDF into a single DataFrame.
    
    Returns:
        df: DataFrame with columns year, month, time, front_lon/lat, envelope boundaries, peak, width
        eke_fields: dict mapping (year, month) -> (eke_2d, lat_1d, lon_1d)
        ke_fields:  dict mapping (year, month) -> (ke_2d,  lat_1d, lon_1d)
                    (may be empty if output files pre-date KE addition)
        coords: dict with 'latitude' and 'longitude' arrays (from the first file, for reference)
    """
    records = []
    eke_fields = {}
    ke_fields = {}
    coords = {}
    n_missing_ke = 0

    years = sorted([y for y in input_dir.iterdir()
                    if y.is_dir() and y.name != ".DS_Store"])

    for year_path in years:
        months = sorted([m for m in year_path.iterdir()
                         if m.is_file() and m.suffix == ".nc"])
        for month_path in months:
            ds = xr.open_dataset(month_path)
            yr = int(year_path.name)
            mo = int(month_path.stem)
            t = pd.Timestamp(yr, mo, 15)  # mid-month

            n = len(ds.front)
            if n == 0:
                continue

            for i in range(n):
                records.append({
                    "year": yr, "month": mo, "time": t,
                    "front_lon": float(ds.front_lon.values[i]),
                    "front_lat": float(ds.front_lat.values[i]),
                    "envelope_lat_south": float(ds.envelope_lat_south.values[i]),
                    "envelope_lat_north": float(ds.envelope_lat_north.values[i]),
                    "peak_lat": float(ds.peak_lat.values[i]),
                    "peak_val": float(ds.peak_val.values[i]),
                    "width_km": float(ds.width_km.values[i]),
                })

            # Store 2-D energy fields alongside their own coordinates
            lat = ds.latitude.values.copy()
            lon = ds.longitude.values.copy()

            eke_fields[(yr, mo)] = (ds["eke"].values.copy(), lat, lon)

            # KE may be absent in files generated before it was added
            if "ke" in ds:
                ke_fields[(yr, mo)] = (ds["ke"].values.copy(), lat, lon)
            else:
                n_missing_ke += 1

            if not coords:
                coords["latitude"] = lat
                coords["longitude"] = lon
            ds.close()

    df = pd.DataFrame(records)
    if not df.empty:
        df["time"] = pd.to_datetime(df["time"])

    if n_missing_ke > 0:
        print(f"⚠️  {n_missing_ke} file(s) missing 'ke' variable — "
              f"KE trends will be unavailable. Re-run Notebook 1 to regenerate outputs.")

    return df, eke_fields, ke_fields, coords


print("Loading data...")
df, eke_fields, ke_fields, coords = load_all_monthly_outputs(INPUT_DIR)

if df.empty:
    print("ERROR: No data found. Run Notebook 1 first.")
else:
    years = sorted(df["year"].unique())
    n_months = df.groupby(["year", "month"]).ngroups
    print(f"Loaded {len(df):,} frontal detections over {n_months} months ({years[0]}–{years[-1]})")
    print(f"EKE fields: {len(eke_fields)}   KE fields: {len(ke_fields)}")


Loading data...
Loaded 1,027,063 frontal detections over 300 months (2001–2025)
EKE fields: 300   KE fields: 300


In [5]:
# Quick summary statistics
if not df.empty:
    print("=== Dataset Summary ===")
    print(f"Total fronts detected: {len(df):,}")
    print(f"Year range: {df['year'].min()} – {df['year'].max()}")
    print(f"Latitude range: [{df['front_lat'].min():.1f}°, {df['front_lat'].max():.1f}°]")
    print(f"Longitude range: [{df['front_lon'].min():.1f}°, {df['front_lon'].max():.1f}°]")
    print(f"\nMean peak EKE: {df['peak_val'].mean():.4f} m²/s²")
    print(f"Mean envelope width: {df['width_km'].mean():.0f} km")
    print(f"\nFronts per month (mean): {len(df) / n_months:.0f}")
    print(f"\nFirst 5 rows:")
    display(df.head())

=== Dataset Summary ===
Total fronts detected: 1,027,063
Year range: 2001 – 2025
Latitude range: [-62.8°, -35.2°]
Longitude range: [-179.9°, 179.9°]

Mean peak EKE: 0.1774 m²/s²
Mean envelope width: 188 km

Fronts per month (mean): 3424

First 5 rows:


,year,month,time,front_lon,front_lat,envelope_lat_south,envelope_lat_north,peak_lat,peak_val,width_km
0,2001,1,2001-01-15,-178.9375,-52.360873,-53.0625,-51.8125,-52.3125,0.141460,166.792390
1,2001,1,2001-01-15,-178.8125,-52.385064,-53.3125,-51.8125,-52.1875,0.141410,166.792390
2,2001,1,2001-01-15,-178.5625,-52.364164,-53.4375,-51.6875,-52.0625,0.140046,194.591122
3,2001,1,2001-01-15,-177.9375,-57.844776,-58.3125,-57.3125,-57.9375,0.098006,111.194927
4,2001,1,2001-01-15,-177.9375,-53.618500,-54.4375,-52.9375,-53.3125,0.071395,166.792390


## 5. ACC Contour Mask (optional)

Mask built from the mean ADT between the −0.6 m and +0.2 m contours (Sokolov & Rintoul 2009), restricting analyses to the dynamically active ACC band instead of a fixed latitude range.

In [6]:
# ── Load mean ADT and build ACC SSH-contour mask ───────────────────────────
# The mean ADT file is produced by Notebook 1 (Pass B).
# Run Notebook 1 first if outputs/mean_adt.nc does not yet exist.

MEAN_ADT_PATH = REPO_ROOT / "outputs" / "mean_adt.nc"
acc_mask = None

if USE_ACC_MASK:
    if not MEAN_ADT_PATH.exists():
        print(f"WARNING: {MEAN_ADT_PATH} not found — ACC mask disabled.\n"
              "Re-run Notebook 1 to generate the mean ADT file.")
        USE_ACC_MASK = False
    else:
        ds_adt  = xr.open_dataset(MEAN_ADT_PATH)
        acc_mask = compute_acc_mask_from_ssh(
            ds_adt['mean_adt'],
            lat=ds_adt.latitude.values,
            lon=ds_adt.longitude.values,
            ssh_south=SSH_ACC_SOUTH,
            ssh_north=SSH_ACC_NORTH,
        )
        ds_adt.close()
        n_acc = int(acc_mask.sum())
        frac  = 100 * n_acc / acc_mask.size
        print(f"ACC SSH mask loaded: {n_acc:,} grid cells ({frac:.1f}% of domain)")
        print(f"  Contours: {SSH_ACC_SOUTH} m ≤ ADT ≤ {SSH_ACC_NORTH} m  "
              f"(Sokolov & Rintoul 2009)")

        # Apply mask to the front DataFrame immediately
        n_before = len(df)
        df = filter_fronts_by_acc_mask(df, acc_mask)
        print(f"\nFront DataFrame: {n_before:,} → {len(df):,} detections "
              f"after ACC masking ({100*len(df)/n_before:.1f}% retained)")
else:
    print("ACC SSH mask: DISABLED (using full LAT_MIN/LAT_MAX bounding box)")


ACC SSH mask loaded: 145,607 grid cells (21.1% of domain)
  Contours: -0.6 m ≤ ADT ≤ 0.2 m  (Sokolov & Rintoul 2009)

Front DataFrame: 1,027,063 → 492,063 detections after ACC masking (47.9% retained)


## 6. Sector-Level Monthly Time Series (Fronts)

Monthly statistics per sector: number of fronts, median/mean frontal latitude, mean peak value, mean envelope width and span.

In [7]:
def compute_sector_timeseries(df, sectors):
    """Aggregate front detections into monthly sector-level statistics."""
    df = df.copy()
    df["sector"] = df["front_lon"].apply(lambda x: assign_sector(x, sectors))
    df["envelope_span"] = df["envelope_lat_north"] - df["envelope_lat_south"]

    groups = df.groupby(["sector", "year", "month"])
    rows = []
    for (sector, yr, mo), grp in groups:
        rows.append({
            "sector": sector, "year": yr, "month": mo,
            "decimal_year": yr + (mo - 0.5) / 12.0,
            "n_fronts": len(grp),
            "mean_lat": grp["front_lat"].mean(),
            "median_lat": grp["front_lat"].median(),
            "std_lat": grp["front_lat"].std(),
            "mean_peak_val": grp["peak_val"].mean(),
            "median_peak_val": grp["peak_val"].median(),
            "mean_width_km": grp["width_km"].mean(),
            "mean_envelope_span": grp["envelope_span"].mean(),
        })
    return pd.DataFrame(rows)


def compute_whole_SO_timeseries(df):
    """Aggregate all fronts per month regardless of sector."""
    df = df.copy()
    df["envelope_span"] = df["envelope_lat_north"] - df["envelope_lat_south"]
    groups = df.groupby(["year", "month"])
    rows = []
    for (yr, mo), grp in groups:
        rows.append({
            "sector": "Whole SO", "year": yr, "month": mo,
            "decimal_year": yr + (mo - 0.5) / 12.0,
            "n_fronts": len(grp),
            "mean_lat": grp["front_lat"].mean(),
            "median_lat": grp["front_lat"].median(),
            "std_lat": grp["front_lat"].std(),
            "mean_peak_val": grp["peak_val"].mean(),
            "median_peak_val": grp["peak_val"].median(),
            "mean_width_km": grp["width_km"].mean(),
            "mean_envelope_span": grp["envelope_span"].mean(),
        })
    return pd.DataFrame(rows)


if not df.empty:
    print("Computing sector time series...")
    ts_sector = compute_sector_timeseries(df, ZHANG_SECTORS)
    ts_whole = compute_whole_SO_timeseries(df)
    ts = pd.concat([ts_sector, ts_whole], ignore_index=True)
    print(f"Sectors: {', '.join(ts['sector'].unique())}")
    print(f"Total rows: {len(ts)}")

Computing sector time series...
Sectors: Atlantic, Indian, Pacific, Whole SO
Total rows: 1200


## 7. Gridded KE and EKE Time Series

Area-weighted (cos-latitude) monthly mean KE and EKE per sector, whole domain, and optionally within the ACC contour mask.

In [8]:
def compute_energy_field_timeseries(fields, field_name, sectors, acc_mask=None):
    """Compute domain-mean and sector-mean of a 2-D energy field.

    Parameters
    ----------
    fields : dict
        Mapping (yr, mo) -> (field_2d, lat_1d, lon_1d).
    field_name : str
        Label used for the output columns, e.g. 'eke' or 'ke'.
    sectors : dict
        Zhang et al. (2021) sector definitions.
    acc_mask : xr.DataArray or None, optional
        2-D boolean mask (True = inside ACC band) produced by
        ``compute_acc_mask_from_ssh``.  When supplied, grid cells outside
        the ACC are set to NaN before computing area-weighted means, so
        both the Whole-SO and sector averages reflect only the ACC.
        Follows the Sokolov & Rintoul (2009) SSH-contour approach.

    Returns
    -------
    pd.DataFrame with columns: sector, year, month, decimal_year,
        mean_{field_name}, max_{field_name}.
    """
    col_mean = f"mean_{field_name}"
    col_max  = f"max_{field_name}"

    # Pre-compute a single aligned acc_mask numpy array if needed.
    # Done once (outside the loop) using the first available field's grid.
    _acc_np_cache = {}   # maps id(lat.tobytes()) -> 2D bool array

    def _get_acc_np(lat, lon):
        key = (lat.tobytes(), lon.tobytes())
        if key not in _acc_np_cache:
            da_dummy = xr.DataArray(
                np.zeros((len(lat), len(lon))),
                dims=['latitude', 'longitude'],
                coords={'latitude': lat, 'longitude': lon},
            )
            aligned = acc_mask.reindex_like(da_dummy, method='nearest', tolerance=0.2)
            _acc_np_cache[key] = aligned.values.astype(bool)
        return _acc_np_cache[key]

    rows = []
    for (yr, mo), (data2d, lat, lon) in sorted(fields.items()):
        # ── Apply ACC SSH-contour mask ──────────────────────────────────────
        if acc_mask is not None:
            data2d = np.where(_get_acc_np(lat, lon), data2d, np.nan)

        cos_lat = np.cos(np.radians(lat))
        valid = np.isfinite(data2d)
        weights = np.broadcast_to(cos_lat[:, None], data2d.shape)
        w = np.where(valid, weights, 0.0)
        mean_val = np.nansum(data2d * w) / np.nansum(w) if np.nansum(w) > 0 else np.nan
        max_val  = np.nanmax(data2d) if valid.any() else np.nan

        rows.append({
            "sector": "Whole SO", "year": yr, "month": mo,
            "decimal_year": yr + (mo - 0.5) / 12.0,
            col_mean: mean_val, col_max: max_val,
        })

        for sname, (lo, hi) in sectors.items():
            mask_lon = np.array([lon_in_sector(l, lo, hi) for l in lon])
            sector_data = data2d[:, mask_lon]
            sector_weights = np.broadcast_to(cos_lat[:, None], sector_data.shape)
            valid_s = np.isfinite(sector_data)
            ws = np.where(valid_s, sector_weights, 0.0)
            smean = np.nansum(sector_data * ws) / np.nansum(ws) if np.nansum(ws) > 0 else np.nan
            smax  = np.nanmax(sector_data) if valid_s.any() else np.nan

            rows.append({
                "sector": sname, "year": yr, "month": mo,
                "decimal_year": yr + (mo - 0.5) / 12.0,
                col_mean: smean, col_max: smax,
            })

    return pd.DataFrame(rows)


if not df.empty:
    print("Computing area-weighted mean EKE and KE from gridded fields...")
    if acc_mask is not None:
        print("  ACC SSH mask applied to gridded fields.")

    eke_ts = compute_energy_field_timeseries(eke_fields, "eke", ZHANG_SECTORS,
                                             acc_mask=acc_mask)
    ke_ts  = compute_energy_field_timeseries(ke_fields,  "ke",  ZHANG_SECTORS,
                                             acc_mask=acc_mask)

    # Merge KE columns into EKE time-series DataFrame
    energy_ts = eke_ts.merge(
        ke_ts, on=["sector", "year", "month", "decimal_year"], how="outer"
    )

    print(f"{len(energy_ts)} sector×month records  "
          f"(columns: {', '.join(c for c in energy_ts.columns if c.startswith('me') or c.startswith('ma'))})")
    display(energy_ts.head(10))


Computing area-weighted mean EKE and KE from gridded fields...
  ACC SSH mask applied to gridded fields.
1200 sector×month records  (columns: mean_eke, max_eke, mean_ke, max_ke)


,sector,year,month,decimal_year,mean_eke,max_eke,mean_ke,max_ke
0,Atlantic,2001,1,2001.041667,0.023712,0.348498,0.028259,0.539334
1,Atlantic,2001,2,2001.125000,0.022131,0.382963,0.028667,0.448443
2,Atlantic,2001,3,2001.208333,0.024940,0.800843,0.031449,0.804264
3,Atlantic,2001,4,2001.291667,0.023800,0.575060,0.030646,0.873529
4,Atlantic,2001,5,2001.375000,0.022986,0.531727,0.028673,0.531954
5,Atlantic,2001,6,2001.458333,0.023811,0.497560,0.029314,0.447583
6,Atlantic,2001,7,2001.541667,0.028833,0.494711,0.030667,0.501499
7,Atlantic,2001,8,2001.625000,0.026613,0.284857,0.028285,0.487142
8,Atlantic,2001,9,2001.708333,0.025779,0.355056,0.028887,0.486163
9,Atlantic,2001,10,2001.791667,0.027518,0.543736,0.030106,0.589237


## 8. Fit Trends

Theil–Sen + Modified Mann–Kendall per sector for each variable (frontal latitude, envelope width, peak value, front count, mean KE, mean EKE).

In [9]:
def fit_all_trends(ts, variable, groupby="sector"):
    """Fit Theil-Sen + modified Mann-Kendall trends per group."""
    results = []
    for name, grp in ts.groupby(groupby):
        x = grp["decimal_year"].values
        y = grp[variable].values

        ts_sl, ts_ic, ts_se, ts_lo, ts_hi = theil_sen_slope(x, y)
        mk_tau, mk_p = mann_kendall(y)

        results.append({
            groupby: name, "variable": variable,
            "n_points": int(np.isfinite(y).sum()),
            "ts_slope": ts_sl, "ts_intercept": ts_ic, "ts_stderr": ts_se,
            "ts_slope_lo95": ts_lo, "ts_slope_hi95": ts_hi,
            "mk_tau": mk_tau, "mk_pvalue": mk_p,
        })

    return pd.DataFrame(results)


print("fit_all_trends() defined.")

fit_all_trends() defined.


In [10]:
# Fit trends on all key variables from the front-level time series
if not df.empty:
    variables_to_trend = [
        "median_lat",         # frontal position
        "mean_peak_val",      # peak EKE within envelopes
        "mean_width_km",      # envelope width
        "mean_envelope_span", # latitudinal span of envelopes
        "n_fronts",           # number of detected fronts
    ]

    trend_tables = {}
    for var in variables_to_trend:
        tdf = fit_all_trends(ts, var)
        trend_tables[var] = tdf

        print(f"\n{'─'*70}")
        print(f"  {var}")
        print(f"{'─'*70}")
        for _, row in tdf.iterrows():
            mk_p = row["mk_pvalue"]
            sig = "***" if mk_p < 0.001 else ("**" if mk_p < 0.01 else ("*" if mk_p < 0.05 else ""))
            print(f"  {row['sector']:20s}  Theil-Sen = {row['ts_slope']:+.4e}/yr  "
                  f"MK p = {mk_p:.3f} {sig}  "
                  f"MK tau = {row['mk_tau']:+.3f}")

    # Fit trends on gridded EKE and KE
    for energy_var, label in [("mean_eke", "EKE"), ("mean_ke", "KE")]:
        energy_trend = fit_all_trends(energy_ts, energy_var)
        trend_tables[f"{energy_var}_gridded"] = energy_trend

        print(f"\n{'─'*70}")
        print(f"  {energy_var} (gridded field, area-weighted) — {label}")
        print(f"{'─'*70}")
        for _, row in energy_trend.iterrows():
            mk_p = row["mk_pvalue"]
            sig = "***" if mk_p < 0.001 else ("**" if mk_p < 0.01 else ("*" if mk_p < 0.05 else ""))
            print(f"  {row['sector']:20s}  Theil-Sen = {row['ts_slope']:+.4e}/yr  "
                  f"MK p = {mk_p:.3f} {sig}")


──────────────────────────────────────────────────────────────────────
  median_lat
──────────────────────────────────────────────────────────────────────
  Atlantic              Theil-Sen = +1.3452e-02/yr  MK p = 0.000 ***  MK tau = +0.156
  Indian                Theil-Sen = -2.0934e-02/yr  MK p = 0.000 ***  MK tau = -0.247
  Pacific               Theil-Sen = -8.6069e-03/yr  MK p = 0.000 ***  MK tau = -0.162
  Whole SO              Theil-Sen = -6.7456e-03/yr  MK p = 0.093   MK tau = -0.065

──────────────────────────────────────────────────────────────────────
  mean_peak_val
──────────────────────────────────────────────────────────────────────
  Atlantic              Theil-Sen = +3.5778e-04/yr  MK p = 0.002 **  MK tau = +0.120
  Indian                Theil-Sen = +3.6666e-04/yr  MK p = 0.000 ***  MK tau = +0.155
  Pacific               Theil-Sen = +5.9941e-04/yr  MK p = 0.000 ***  MK tau = +0.207
  Whole SO              Theil-Sen = +4.3747e-04/yr  MK p = 0.000 ***  MK tau = +0.254



## 9. Eddy Saturation Test

Per sector, compare the EKE trend against the KE trend: under **eddy saturation**, increased wind forcing energises the eddy field (EKE ↑) while the mean flow — and hence transport — stays approximately constant (KE trend ≈ 0 or ≪ EKE trend).


In [11]:
def eddy_saturation_test(ts, energy_ts):
    """Test the eddy saturation hypothesis per sector.

    Diagnosis
    ---------
    - EKE increasing (MK p < 0.05, positive Theil-Sen slope)
    - KE not significantly increasing (MK p >= 0.05 or non-positive slope)
    - Frontal latitude stationary (MK p >= 0.05)
    """
    alpha = 0.05
    results = []
    sectors = ts["sector"].unique()

    for sector in sectors:
        grp_ts = ts[ts["sector"] == sector].sort_values("decimal_year")
        grp_ene = energy_ts[energy_ts["sector"] == sector].sort_values("decimal_year")

        # Frontal latitude trend
        x_lat = grp_ts["decimal_year"].values
        y_lat = grp_ts["median_lat"].values
        sl_lat, _, _, _, _ = theil_sen_slope(x_lat, y_lat)
        _, pv_lat = mann_kendall(y_lat)

        # Peak EKE at fronts
        y_peak = grp_ts["mean_peak_val"].values
        sl_peak, _, _, _, _ = theil_sen_slope(x_lat, y_peak)
        _, pv_peak = mann_kendall(y_peak)

        # Gridded EKE
        if not grp_ene.empty and "mean_eke" in grp_ene.columns:
            x_ene = grp_ene["decimal_year"].values
            y_eke = grp_ene["mean_eke"].values
            sl_eke, _, _, _, _ = theil_sen_slope(x_ene, y_eke)
            _, pv_eke = mann_kendall(y_eke)
        else:
            sl_eke, pv_eke = np.nan, np.nan

        # Gridded KE (mean kinetic energy)
        if not grp_ene.empty and "mean_ke" in grp_ene.columns:
            y_ke = grp_ene["mean_ke"].values
            sl_ke, _, _, _, _ = theil_sen_slope(x_ene, y_ke)
            _, pv_ke = mann_kendall(y_ke)
        else:
            sl_ke, pv_ke = np.nan, np.nan

        # Significance flags
        peak_eke_sig = (pv_peak < alpha) and (sl_peak > 0) if np.isfinite(pv_peak) else False
        eke_sig = (pv_eke < alpha) and (sl_eke > 0) if np.isfinite(pv_eke) else False
        ke_sig = (pv_ke < alpha) and (sl_ke > 0) if np.isfinite(pv_ke) else False
        lat_sig = (pv_lat < alpha) if np.isfinite(pv_lat) else False

        # Eddy-saturated: EKE increasing, latitude stable, KE not increasing
        saturated = (peak_eke_sig or eke_sig) and (not lat_sig) and (not ke_sig)

        results.append({
            "sector": sector,
            "peak_eke_trend_per_yr": sl_peak, "peak_eke_pvalue": pv_peak, "peak_eke_significant": peak_eke_sig,
            "mean_eke_trend_per_yr": sl_eke, "mean_eke_pvalue": pv_eke, "mean_eke_significant": eke_sig,
            "mean_ke_trend_per_yr": sl_ke, "mean_ke_pvalue": pv_ke, "mean_ke_significant": ke_sig,
            "lat_trend_per_yr": sl_lat, "lat_pvalue": pv_lat, "lat_significant": lat_sig,
            "eddy_saturated": saturated,
        })

    return pd.DataFrame(results)


if not df.empty:
    print("=" * 70)
    print("EDDY SATURATION TEST")
    print("=" * 70)
    saturation = eddy_saturation_test(ts, energy_ts)

    for _, row in saturation.iterrows():
        status = "SATURATED" if row["eddy_saturated"] else "NOT SATURATED (or insufficient evidence)"
        print(f"\n  {row['sector']:20s}  -> {status}")
        print(f"    Peak EKE trend: {row['peak_eke_trend_per_yr']:+.4e}/yr  (MK p={row['peak_eke_pvalue']:.3f})")
        print(f"    Mean EKE trend: {row['mean_eke_trend_per_yr']:+.4e}/yr  (MK p={row['mean_eke_pvalue']:.3f})")
        print(f"    Mean KE  trend: {row['mean_ke_trend_per_yr']:+.4e}/yr  (MK p={row['mean_ke_pvalue']:.3f})")
        print(f"    Lat trend:      {row['lat_trend_per_yr']:+.4e} deg/yr  (MK p={row['lat_pvalue']:.3f})")

EDDY SATURATION TEST

  Atlantic              -> NOT SATURATED (or insufficient evidence)
    Peak EKE trend: +3.5778e-04/yr  (MK p=0.002)
    Mean EKE trend: +2.1456e-04/yr  (MK p=0.000)
    Mean KE  trend: +1.2472e-04/yr  (MK p=0.000)
    Lat trend:      +1.3452e-02 deg/yr  (MK p=0.000)

  Indian                -> NOT SATURATED (or insufficient evidence)
    Peak EKE trend: +3.6666e-04/yr  (MK p=0.000)
    Mean EKE trend: +1.3159e-04/yr  (MK p=0.000)
    Mean KE  trend: +9.2647e-05/yr  (MK p=0.000)
    Lat trend:      -2.0934e-02 deg/yr  (MK p=0.000)

  Pacific               -> NOT SATURATED (or insufficient evidence)
    Peak EKE trend: +5.9941e-04/yr  (MK p=0.000)
    Mean EKE trend: +2.4573e-04/yr  (MK p=0.000)
    Mean KE  trend: +1.8288e-04/yr  (MK p=0.000)
    Lat trend:      -8.6069e-03 deg/yr  (MK p=0.000)

  Whole SO              -> NOT SATURATED (or insufficient evidence)
    Peak EKE trend: +4.3747e-04/yr  (MK p=0.000)
    Mean EKE trend: +1.9783e-04/yr  (MK p=0.000)
    M

In [12]:
# Display saturation results as a summary table
if not df.empty:
    bool_cols = ["peak_eke_significant", "mean_eke_significant",
                 "mean_ke_significant", "lat_significant", "eddy_saturated"]
    display(saturation[["sector"] + bool_cols].style.applymap(
        lambda v: 'background-color: #d4edda' if v is True and isinstance(v, bool) else 
                  ('background-color: #f8d7da' if v is False and isinstance(v, bool) else ''),
        subset=bool_cols
    ))


/var/folders/ms/8hpcnj2j01zd81g0v_5l2ttw0000gn/T/ipykernel_6912/2690102997.py:5: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(saturation[["sector"] + bool_cols].style.applymap(


,sector,peak_eke_significant,mean_eke_significant,mean_ke_significant,lat_significant,eddy_saturated
0,Atlantic,True,True,True,True,False
1,Indian,True,True,True,True,False
2,Pacific,True,True,True,True,False
3,Whole SO,True,True,True,False,False


## 10. Longitude-Resolved Trends

Theil–Sen + MK trends of frontal latitude and peak value in 30° longitude bands.

In [13]:
def compute_lon_band_trends(df, lon_bands, variable="front_lat"):
    """Compute Theil-Sen and modified MK trends in `variable` per longitude band."""
    df = df.copy()
    df["lon_band"] = df["front_lon"].apply(lambda x: assign_sector(x, lon_bands))
    df["decimal_year"] = df["year"] + (df["month"] - 0.5) / 12.0

    grp = df.groupby(["lon_band", "year", "month"]).agg(
        median_val=(variable, "median"),
        mean_val=(variable, "mean"),
        decimal_year=("decimal_year", "first"),
    ).reset_index()

    results = []
    for band, g in grp.groupby("lon_band"):
        x = g["decimal_year"].values
        y = g["median_val"].values
        ts_sl, _, ts_se, ts_lo, ts_hi = theil_sen_slope(x, y)
        mk_tau, mk_p = mann_kendall(y)

        lo, hi = lon_bands.get(band, (np.nan, np.nan))
        mid_lon = (lo + hi) / 2.0 if not np.isnan(lo) else np.nan

        results.append({
            "lon_band": band, "mid_lon": mid_lon, "n_months": len(g),
            "ts_slope": ts_sl, "ts_stderr": ts_se,
            "ts_slope_lo95": ts_lo, "ts_slope_hi95": ts_hi,
            "mk_tau": mk_tau, "mk_pvalue": mk_p,
        })

    return pd.DataFrame(results)


if not df.empty:
    print("Computing longitude-resolved trends...")
    lon_trend_lat = compute_lon_band_trends(df, ZHANG_SECTORS, variable="front_lat")
    lon_trend_ke = compute_lon_band_trends(df, ZHANG_SECTORS, variable="peak_val")
    lon_trends = {"front_lat": lon_trend_lat, "peak_val": lon_trend_ke}

    n_sig_lat = (lon_trend_lat["mk_pvalue"] < 0.05).sum()
    n_sig_ke = (lon_trend_ke["mk_pvalue"] < 0.05).sum()
    print(f"  {n_sig_lat}/{len(lon_trend_lat)} lon bands show significant lat trends (MK p<0.05)")
    print(f"  {n_sig_ke}/{len(lon_trend_ke)} lon bands show significant KE trends (MK p<0.05)")

Computing longitude-resolved trends...
  3/3 lon bands show significant lat trends (MK p<0.05)
  3/3 lon bands show significant KE trends (MK p<0.05)


## 11. Save Results

Written to `outputs/trends/`:

- `sector_timeseries.csv` — monthly front statistics per sector (consumed by Notebooks 03b and 07)
- `trends_*.csv` — per-variable trend tables
- `eddy_saturation_test.csv` — saturation test summary
- `lon_band_trends_*.csv` — longitude-resolved trends


In [14]:
if not df.empty:
    # Time series
    ts.to_csv(OUTPUT_DIR / "sector_timeseries.csv", index=False)
    # NOTE: energy_field_timeseries.csv is written by scripts/ke_trends_analysis.py
    # (canonical producer, "region" column); it is intentionally NOT overwritten here.

    # Trend tables (includes mean_eke_gridded and mean_ke_gridded)
    for name, tbl in trend_tables.items():
        tbl.to_csv(OUTPUT_DIR / f"trends_{name}.csv", index=False)

    # Eddy saturation summary
    saturation.to_csv(OUTPUT_DIR / "eddy_saturation_test.csv", index=False)

    # Longitude-resolved trends
    for name, tbl in lon_trends.items():
        tbl.to_csv(OUTPUT_DIR / f"lon_band_trends_{name}.csv", index=False)

    print(f"All results saved to {OUTPUT_DIR}/")
    print(f"\nFiles:")
    for f in sorted(OUTPUT_DIR.glob("*.csv")):
        print(f"  {f.name}")


All results saved to outputs/trends/

Files:
  drake_passage_energy_timeseries.csv
  drake_passage_geostrophic_transport_observations.csv
  drake_passage_total_transport_observations.csv
  drake_passage_transport.csv
  drake_passage_trends.csv
  eddy_saturation_test.csv
  energy_field_timeseries.csv
  energy_field_timeseries_two_sat.csv
  lon_band_trends_front_lat.csv
  lon_band_trends_peak_val.csv
  region_timeseries.csv
  sector_timeseries.csv
  trend_bars_two_sat.csv
  trend_comparison_full_vs_since2016.csv
  trends_mean_eke_gridded.csv
  trends_mean_envelope_span.csv
  trends_mean_ke_gridded.csv
  trends_mean_peak_val.csv
  trends_mean_width_km.csv
  trends_median_lat.csv
  trends_median_lat_reduced_ci.csv
  trends_n_fronts.csv
  wind_stress_sector_timeseries.csv
  wind_stress_timeseries.csv
  wind_stress_trends.csv
  wind_transport_correlation_results.csv
  wind_transport_summary_stats.csv


---

## Summary

This notebook turns the monthly front detections into sector-level time series and non-parametric trend estimates, tests the eddy saturation hypothesis, and writes all tables to `outputs/trends/`.

Continue with **Notebook 03a** (KE/EKE article figure) and **Notebook 03b** (fronts article figure).
